In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('../data/raw/train_df.csv')
df1 = pd.read_csv('../data/processed/music_accounts.csv')
df2 = pd.read_csv('../data/processed/music_track_events.csv')

In [3]:
df.head()

,row_id,user_id,music_app_user_id,age,education,spending_capacity_band,home_city,home_lat,home_lon,stated_favourite_genre,...,sale_opened_at,prediction_at,ticket_price,booking_fee,discount_pct,event_page_visits_7d,wishlisted,travel_search_city,travel_search_arrival,buy
0,R2F019957,U2CFDCAAD,M1B01D2D7,23,postgraduate,2,Nantes,47.218,-1.554,pop,...,2026-09-28T00:00:00Z,2026-10-01T00:00:00Z,75.0,9.50,0,0.0,1,NaN,NaN,0
1,R0804D93A,U1FC01010,M1B2C63CC,51,undergraduate,5,Bordeaux,44.838,-0.579,pop,...,2026-09-28T00:00:00Z,2026-10-01T00:00:00Z,85.0,10.10,0,3.0,0,NaN,NaN,1
2,R367FE81E,U0C605712,M3AEF638D,32,secondary,1,Lyon,45.764,4.835,classical,...,2026-09-28T00:00:00Z,2026-10-01T00:00:00Z,175.0,15.50,20,1.0,1,NaN,NaN,0
3,R344BBE29,U25D1DD8A,M00C580ED,24,secondary,2,Paris,48.857,2.352,jazz_funk,...,2026-09-28T00:00:00Z,2026-10-01T00:00:00Z,84.0,10.04,0,1.0,1,NaN,NaN,1
4,R004ABDEB,U1600BC92,M29AC37E6,21,secondary,3,Bordeaux,44.838,-0.579,NaN,...,2026-09-28T00:00:00Z,2026-10-01T00:00:00Z,95.0,10.70,0,0.0,0,Rennes,2026-12-28,1


In [4]:
df1.head()

,music_app_user_id,activity_status,total_sessions,total_track_events
0,M1213C172,active,53,82
1,M2B19FDA8,active,61,100
2,M24E86338,active,42,54
3,M27ADBF54,active,32,39
4,M2136A90A,active,36,47


In [5]:
df2.head()

,music_app_user_id,activity_status,session_id,played_at,artist_id,genre,track_id,played_ms,track_ms,source,saved
0,M1213C172,active,S001,2026-09-30T21:22:16Z,A11,alternative,A11_6,132490,160000,autoplay,False
1,M1213C172,active,S002,2026-09-29T00:22:53Z,A08,jazz_funk,A08_6,198000,198000,autoplay,False
2,M1213C172,active,S003,2026-09-28T00:42:34Z,A08,jazz_funk,A08_12,223698,273000,search,True
3,M1213C172,active,S003,2026-09-28T17:59:57Z,A11,alternative,A11_12,205737,232000,search,False
4,M1213C172,active,S004,2026-09-27T21:36:03Z,A16,metal,A16_10,180541,196000,search,False


In [6]:
df = df.merge(
    df1,
    on="music_app_user_id",
    how="left",
    validate="many_to_one"
)

In [9]:
df.shape

(30000, 35)

In [10]:
df.columns

Index(['row_id', 'user_id', 'music_app_user_id', 'age', 'education',
       'spending_capacity_band', 'home_city', 'home_lat', 'home_lon',
       'stated_favourite_genre', 'past_purchase_count', 'first_timer',
       'typical_ticket_spend', 'usual_booking_lead_days', 'concert_id',
       'artist_id', 'artist_name', 'genres', 'venue_city', 'venue_lat',
       'venue_lon', 'concert_at', 'sale_opened_at', 'prediction_at',
       'ticket_price', 'booking_fee', 'discount_pct', 'event_page_visits_7d',
       'wishlisted', 'travel_search_city', 'travel_search_arrival', 'buy',
       'activity_status', 'total_sessions', 'total_track_events'],
      dtype='str')

In [12]:
df2.columns

Index(['music_app_user_id', 'activity_status', 'session_id', 'played_at',
       'artist_id', 'genre', 'track_id', 'played_ms', 'track_ms', 'source',
       'saved'],
      dtype='str')

In [13]:
df2.shape

(551901, 11)

In [14]:
df2['artist_id'].nunique()

16

In [15]:
df2['artist_id'].value_counts()

artist_id
A03    41902
A01    37232
A12    36690
A11    35653
A05    35412
A14    35263
A13    35083
A06    34114
A07    33794
A08    33664
A02    32941
A15    32730
A10    32722
A16    32645
A09    32123
A04    29933
Name: count, dtype: int64

In [16]:
df2["genre"].nunique()

11

In [17]:
df2["genre"].value_counts()

genre
pop             80039
hip_hop         59992
rnb_soul        58850
alternative     53773
pop_rock        50113
psychedelic     47381
jazz_funk       46600
folk_country    45572
electronic      44206
classical       32730
metal           32645
Name: count, dtype: int64

In [18]:
user_music_features = (
    df2.groupby("music_app_user_id")
    .agg(
        unique_artists=("artist_id", "nunique"),
        unique_genres=("genre", "nunique"),
        saved_tracks=("saved", "sum"),
        saved_rate=("saved", "mean"),
        autoplay_count=("source", lambda x: (x == "autoplay").sum()),
        search_count=("source", lambda x: (x == "search").sum())
    )
    .reset_index()
)

In [19]:
print(user_music_features.shape)
display(user_music_features.head())

(8746, 7)


,music_app_user_id,unique_artists,unique_genres,saved_tracks,saved_rate,autoplay_count,search_count
0,M000011A5,13,10,14,0.285714,19,30
1,M00001B55,11,10,30,0.300000,32,68
2,M00045894,9,6,18,0.409091,15,29
3,M0004CCB1,13,10,31,0.378049,32,50
4,M000704A9,13,11,12,0.292683,15,26


In [20]:
user_artist_features = (
    df2.groupby(["music_app_user_id", "artist_id"])
    .agg(
        artist_play_count=("track_id", "count"),
        artist_saved_count=("saved", "sum"),
        artist_saved_rate=("saved", "mean"),
        avg_completion=("played_ms", "mean")
    )
    .reset_index()
)

In [21]:
print(user_artist_features.shape)
user_artist_features.head(10)

(99054, 6)


,music_app_user_id,artist_id,artist_play_count,artist_saved_count,artist_saved_rate,avg_completion
0,M000011A5,A01,4,1,0.250000,148041.500000
1,M000011A5,A03,5,2,0.400000,189794.400000
2,M000011A5,A04,7,1,0.142857,169213.428571
3,M000011A5,A05,2,1,0.500000,117355.000000
4,M000011A5,A06,3,0,0.000000,105046.333333
5,M000011A5,A08,2,0,0.000000,145258.500000
6,M000011A5,A09,3,1,0.333333,197531.000000
7,M000011A5,A10,1,0,0.000000,88308.000000
8,M000011A5,A11,13,7,0.538462,187238.846154
9,M000011A5,A12,4,0,0.000000,163879.750000


In [23]:
print("Zero track durations:",
      (df2["track_ms"] <= 0).sum())

print("Missing track durations:",
      df2["track_ms"].isna().sum())

print("Playback longer than track:",
      (df2["played_ms"] > df2["track_ms"]).sum())

Zero track durations: 0
Missing track durations: 0
Playback longer than track: 0


In [24]:
df2["completion_ratio"] = (
    df2["played_ms"] / df2["track_ms"]
)

In [25]:
user_artist_features = (
    df2.groupby(["music_app_user_id", "artist_id"])
    .agg(
        artist_play_count=("track_id", "count"),
        artist_saved_count=("saved", "sum"),
        artist_saved_rate=("saved", "mean"),
        avg_completion_rate=("completion_ratio", "mean")
    )
    .reset_index()
)

In [26]:
display(user_artist_features.head(10))

print(user_artist_features.shape)

print(user_artist_features["avg_completion_rate"].describe())

,music_app_user_id,artist_id,artist_play_count,artist_saved_count,artist_saved_rate,avg_completion_rate
0,M000011A5,A01,4,1,0.250000,0.657216
1,M000011A5,A03,5,2,0.400000,0.931555
2,M000011A5,A04,7,1,0.142857,0.803527
3,M000011A5,A05,2,1,0.500000,0.746223
4,M000011A5,A06,3,0,0.000000,0.459660
5,M000011A5,A08,2,0,0.000000,0.675076
6,M000011A5,A09,3,1,0.333333,0.867640
7,M000011A5,A10,1,0,0.000000,0.573429
8,M000011A5,A11,13,7,0.538462,0.823205
9,M000011A5,A12,4,0,0.000000,0.859011


(99054, 6)
count    99054.000000
mean         0.616847
std          0.268596
min          0.020006
25%          0.444261
50%          0.689527
75%          0.837589
max          1.000000
Name: avg_completion_rate, dtype: float64


In [27]:
df = df.merge(
    user_artist_features,
    on=["music_app_user_id", "artist_id"],
    how="left",
    validate="many_to_one"
)

In [28]:
df.shape

(30000, 39)

In [29]:
df[
    [
        "music_app_user_id",
        "artist_id",
        "artist_play_count",
        "artist_saved_rate",
        "avg_completion_rate"
    ]
].head()

,music_app_user_id,artist_id,artist_play_count,artist_saved_rate,avg_completion_rate
0,M1B01D2D7,A04,NaN,NaN,NaN
1,M1B2C63CC,A02,8.0,0.500000,0.837594
2,M3AEF638D,A03,1.0,0.000000,0.131795
3,M00C580ED,A01,2.0,0.000000,0.115896
4,M29AC37E6,A05,3.0,0.333333,0.901497


In [30]:
user_genre_features = (
    df2.groupby(["music_app_user_id", "genre"])
    .agg(
        genre_play_count=("track_id", "count"),
        genre_saved_count=("saved", "sum"),
        genre_saved_rate=("saved", "mean"),
        avg_genre_completion=("completion_ratio", "mean")
    )
    .reset_index()
)

user_genre_features.duplicated(
    ["music_app_user_id", "genre"]
).sum()

np.int64(0)

In [31]:
df[["concert_id", "artist_id", "genres"]].drop_duplicates()

,concert_id,artist_id,genres
0,C04,A04,pop|alternative|electronic|psychedelic
1,C02,A02,pop|pop_rock|alternative
2,C03,A03,pop|pop_rock|alternative|folk_country
3,C01,A01,hip_hop|rnb_soul|jazz_funk|pop
4,C05,A05,hip_hop|rnb_soul|jazz_funk|pop|alternative


In [32]:
concert_genres = (
    df[["concert_id", "artist_id", "genres"]]
    .drop_duplicates()
    .assign(genre=lambda x: x["genres"].str.split("|", regex=False))
    .explode("genre")
    .drop(columns="genres")
    .reset_index(drop=True)
)

display(concert_genres)

,concert_id,artist_id,genre
0,C04,A04,pop
1,C04,A04,alternative
2,C04,A04,electronic
3,C04,A04,psychedelic
4,C02,A02,pop
5,C02,A02,pop_rock
6,C02,A02,alternative
7,C03,A03,pop
8,C03,A03,pop_rock
9,C03,A03,alternative


In [33]:
user_genre_features = (
    df2.groupby(["music_app_user_id", "genre"])
    .agg(
        genre_play_count=("genre", "size"),
        genre_saved_count=("saved", "sum"),
        genre_saved_rate=("saved", "mean"),
        genre_avg_completion=("completion_ratio", "mean")
    )
    .reset_index()
)

display(user_genre_features.head(10))
print(user_genre_features.shape)

,music_app_user_id,genre,genre_play_count,genre_saved_count,genre_saved_rate,genre_avg_completion
0,M000011A5,alternative,13,7,0.538462,0.823205
1,M000011A5,classical,1,1,1.000000,0.083467
2,M000011A5,electronic,6,1,0.166667,0.815793
3,M000011A5,folk_country,2,0,0.000000,0.464941
4,M000011A5,hip_hop,5,0,0.000000,0.445067
5,M000011A5,jazz_funk,4,1,0.250000,0.783166
6,M000011A5,pop,8,3,0.375000,0.877772
7,M000011A5,pop_rock,2,0,0.000000,0.786714
8,M000011A5,psychedelic,6,0,0.000000,0.728084
9,M000011A5,rnb_soul,2,1,0.500000,0.746223


(78487, 6)


In [34]:
print(user_genre_features.shape)

print(
    user_genre_features
    .groupby("genre")["genre_play_count"]
    .sum()
    .sort_values(ascending=False)
)

(78487, 6)
genre
pop             80039
hip_hop         59992
rnb_soul        58850
alternative     53773
pop_rock        50113
psychedelic     47381
jazz_funk       46600
folk_country    45572
electronic      44206
classical       32730
metal           32645
Name: genre_play_count, dtype: int64


In [35]:
concert_genres = (
    df[["concert_id", "artist_id", "genres"]]
    .drop_duplicates()
    .assign(genre=lambda x: x["genres"].str.split("|"))
    .explode("genre")
    [["concert_id", "artist_id", "genre"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

display(concert_genres)

,concert_id,artist_id,genre
0,C04,A04,pop
1,C04,A04,alternative
2,C04,A04,electronic
3,C04,A04,psychedelic
4,C02,A02,pop
5,C02,A02,pop_rock
6,C02,A02,alternative
7,C03,A03,pop
8,C03,A03,pop_rock
9,C03,A03,alternative


In [36]:
user_concert_genres = concert_genres.merge(
    user_genre_features,
    on="genre",
    how="left"
)

display(user_concert_genres.head())
print(user_concert_genres.shape)

,concert_id,artist_id,genre,music_app_user_id,genre_play_count,genre_saved_count,genre_saved_rate,genre_avg_completion
0,C04,A04,pop,M000011A5,8,3,0.375000,0.877772
1,C04,A04,pop,M00001B55,42,17,0.404762,0.820502
2,C04,A04,pop,M00045894,8,3,0.375000,0.758150
3,C04,A04,pop,M0004CCB1,10,3,0.300000,0.885721
4,C04,A04,pop,M000704A9,7,1,0.142857,0.601885


(154044, 8)


In [38]:
all_user_concert_genres = df1[["music_app_user_id"]].merge(
    concert_genres,
    how="cross"
)

print(all_user_concert_genres.shape)
display(all_user_concert_genres.head())

(188640, 4)


,music_app_user_id,concert_id,artist_id,genre
0,M1213C172,C04,A04,pop
1,M1213C172,C04,A04,alternative
2,M1213C172,C04,A04,electronic
3,M1213C172,C04,A04,psychedelic
4,M1213C172,C02,A02,pop


In [39]:
user_concert_genres = all_user_concert_genres.merge(
    user_genre_features,
    on=["music_app_user_id", "genre"],
    how="left"
)

print(user_concert_genres.shape)
display(user_concert_genres.head())

(188640, 8)


,music_app_user_id,concert_id,artist_id,genre,genre_play_count,genre_saved_count,genre_saved_rate,genre_avg_completion
0,M1213C172,C04,A04,pop,1.0,0.0,0.000000,0.794664
1,M1213C172,C04,A04,alternative,51.0,20.0,0.392157,0.851861
2,M1213C172,C04,A04,electronic,4.0,0.0,0.000000,0.573137
3,M1213C172,C04,A04,psychedelic,1.0,1.0,1.000000,0.809036
4,M1213C172,C02,A02,pop,1.0,0.0,0.000000,0.794664


In [40]:
user_concert_genres[
    ["genre_play_count", "genre_saved_count"]
] = user_concert_genres[
    ["genre_play_count", "genre_saved_count"]
].fillna(0)

In [41]:
user_concert_features = (
    user_concert_genres
    .groupby(
        ["music_app_user_id", "concert_id", "artist_id"]
    )
    .agg(
        concert_genre_total_plays=("genre_play_count", "sum"),
        concert_genre_total_saved=("genre_saved_count", "sum"),
        avg_genre_plays=("genre_play_count", "mean"),
        max_genre_plays=("genre_play_count", "max")
    )
    .reset_index()
)

print(user_concert_features.shape)
display(user_concert_features.head())

(47160, 7)


,music_app_user_id,concert_id,artist_id,concert_genre_total_plays,concert_genre_total_saved,avg_genre_plays,max_genre_plays
0,M000011A5,C01,A01,19.0,5.0,4.750000,8.0
1,M000011A5,C02,A02,23.0,10.0,7.666667,13.0
2,M000011A5,C03,A03,25.0,10.0,6.250000,13.0
3,M000011A5,C04,A04,33.0,11.0,8.250000,13.0
4,M000011A5,C05,A05,32.0,12.0,6.400000,13.0


In [42]:
enriched_df = df.merge(
    user_concert_features,
    on=["music_app_user_id", "concert_id"],
    how="left",
    validate="many_to_one"
)

print("Original shape:", df.shape)
print("New shape:", enriched_df.shape)
display(enriched_df.head())

Original shape: (30000, 39)
New shape: (30000, 44)


,row_id,user_id,music_app_user_id,age,education,spending_capacity_band,home_city,home_lat,home_lon,stated_favourite_genre,...,total_track_events,artist_play_count,artist_saved_count,artist_saved_rate,avg_completion_rate,artist_id_y,concert_genre_total_plays,concert_genre_total_saved,avg_genre_plays,max_genre_plays
0,R2F019957,U2CFDCAAD,M1B01D2D7,23,postgraduate,2,Nantes,47.218,-1.554,pop,...,59.0,NaN,NaN,NaN,NaN,A04,25.0,7.0,6.250000,24.0
1,R0804D93A,U1FC01010,M1B2C63CC,51,undergraduate,5,Bordeaux,44.838,-0.579,pop,...,45.0,8.0,4.0,0.500000,0.837594,A02,22.0,9.0,7.333333,10.0
2,R367FE81E,U0C605712,M3AEF638D,32,secondary,1,Lyon,45.764,4.835,classical,...,45.0,1.0,0.0,0.000000,0.131795,A03,6.0,0.0,1.500000,3.0
3,R344BBE29,U25D1DD8A,M00C580ED,24,secondary,2,Paris,48.857,2.352,jazz_funk,...,87.0,2.0,0.0,0.000000,0.115896,A01,46.0,20.0,11.500000,27.0
4,R004ABDEB,U1600BC92,M29AC37E6,21,secondary,3,Bordeaux,44.838,-0.579,NaN,...,46.0,3.0,1.0,0.333333,0.901497,A05,29.0,13.0,5.800000,16.0


In [55]:
enriched_df.columns[
    enriched_df.columns.str.contains("artist_id")
].tolist()

['artist_id']

In [58]:
enriched_df.to_csv(
    "../data/processed/enriched_train.csv",
    index=False
)

In [56]:
enriched_df.columns

Index(['row_id', 'user_id', 'music_app_user_id', 'age', 'education',
       'spending_capacity_band', 'home_city', 'home_lat', 'home_lon',
       'stated_favourite_genre', 'past_purchase_count', 'first_timer',
       'typical_ticket_spend', 'usual_booking_lead_days', 'concert_id',
       'artist_id', 'artist_name', 'genres', 'venue_city', 'venue_lat',
       'venue_lon', 'concert_at', 'sale_opened_at', 'prediction_at',
       'ticket_price', 'booking_fee', 'discount_pct', 'event_page_visits_7d',
       'wishlisted', 'travel_search_city', 'travel_search_arrival', 'buy',
       'activity_status', 'total_sessions', 'total_track_events',
       'artist_play_count', 'artist_saved_count', 'artist_saved_rate',
       'avg_completion_rate', 'concert_genre_total_plays',
       'concert_genre_total_saved', 'avg_genre_plays', 'max_genre_plays'],
      dtype='str')

In [57]:
enriched_df.shape

(30000, 43)

In [95]:
enriched_df.head(6)

,row_id,user_id,music_app_user_id,age,education,spending_capacity_band,home_city,home_lat,home_lon,stated_favourite_genre,...,total_sessions,total_track_events,artist_play_count,artist_saved_count,artist_saved_rate,avg_completion_rate,concert_genre_total_plays,concert_genre_total_saved,avg_genre_plays,max_genre_plays
0,R2F019957,U2CFDCAAD,M1B01D2D7,23,postgraduate,2,Nantes,47.218,-1.554,pop,...,42.0,59.0,NaN,NaN,NaN,NaN,25.0,7.0,6.250000,24.0
1,R0804D93A,U1FC01010,M1B2C63CC,51,undergraduate,5,Bordeaux,44.838,-0.579,pop,...,38.0,45.0,8.0,4.0,0.500000,0.837594,22.0,9.0,7.333333,10.0
2,R367FE81E,U0C605712,M3AEF638D,32,secondary,1,Lyon,45.764,4.835,classical,...,36.0,45.0,1.0,0.0,0.000000,0.131795,6.0,0.0,1.500000,3.0
3,R344BBE29,U25D1DD8A,M00C580ED,24,secondary,2,Paris,48.857,2.352,jazz_funk,...,51.0,87.0,2.0,0.0,0.000000,0.115896,46.0,20.0,11.500000,27.0
4,R004ABDEB,U1600BC92,M29AC37E6,21,secondary,3,Bordeaux,44.838,-0.579,NaN,...,34.0,46.0,3.0,1.0,0.333333,0.901497,29.0,13.0,5.800000,16.0
5,R366DD8C3,U20C885A7,M28AC108F,44,secondary,3,Paris,48.857,2.352,psychedelic,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,0.000000,0.0


In [97]:
#Will remove unnecessary columns as of now like lat-long
enriched_df[enriched_df['user_id']=='U2CFDCAAD'].head(10)

,row_id,user_id,music_app_user_id,age,education,spending_capacity_band,home_city,home_lat,home_lon,stated_favourite_genre,...,total_sessions,total_track_events,artist_play_count,artist_saved_count,artist_saved_rate,avg_completion_rate,concert_genre_total_plays,concert_genre_total_saved,avg_genre_plays,max_genre_plays
0,R2F019957,U2CFDCAAD,M1B01D2D7,23,postgraduate,2,Nantes,47.218,-1.554,pop,...,42.0,59.0,NaN,NaN,NaN,NaN,25.0,7.0,6.250000,24.0
881,R00DF6039,U2CFDCAAD,M1B01D2D7,23,postgraduate,2,Nantes,47.218,-1.554,pop,...,42.0,59.0,1.0,0.0,0.000000,0.493966,29.0,6.0,7.250000,24.0
5932,R2E3A8D40,U2CFDCAAD,M1B01D2D7,23,postgraduate,2,Nantes,47.218,-1.554,pop,...,42.0,59.0,11.0,5.0,0.454545,0.900456,41.0,15.0,13.666667,24.0
6251,R13F6DB3B,U2CFDCAAD,M1B01D2D7,23,postgraduate,2,Nantes,47.218,-1.554,pop,...,42.0,59.0,1.0,0.0,0.000000,0.565091,30.0,7.0,6.000000,24.0
26201,R2E96D09B,U2CFDCAAD,M1B01D2D7,23,postgraduate,2,Nantes,47.218,-1.554,pop,...,42.0,59.0,2.0,0.0,0.000000,0.846553,43.0,15.0,10.750000,24.0


In [98]:
enriched_df.columns.tolist()

['row_id',
 'user_id',
 'music_app_user_id',
 'age',
 'education',
 'spending_capacity_band',
 'home_city',
 'home_lat',
 'home_lon',
 'stated_favourite_genre',
 'past_purchase_count',
 'first_timer',
 'typical_ticket_spend',
 'usual_booking_lead_days',
 'concert_id',
 'artist_id',
 'artist_name',
 'genres',
 'venue_city',
 'venue_lat',
 'venue_lon',
 'concert_at',
 'sale_opened_at',
 'prediction_at',
 'ticket_price',
 'booking_fee',
 'discount_pct',
 'event_page_visits_7d',
 'wishlisted',
 'travel_search_city',
 'travel_search_arrival',
 'buy',
 'activity_status',
 'total_sessions',
 'total_track_events',
 'artist_play_count',
 'artist_saved_count',
 'artist_saved_rate',
 'avg_completion_rate',
 'concert_genre_total_plays',
 'concert_genre_total_saved',
 'avg_genre_plays',
 'max_genre_plays']

In [104]:
import importlib
import feature_engineering
import preprocessing

importlib.reload(feature_engineering)
importlib.reload(preprocessing)
import feature_engineering
import preprocessing
enriched_df = feature_engineering.build_ifeatures(enriched_df)
enriched_df = preprocessing.remove_cols(enriched_df)

In [105]:
enriched_df.head()

,row_id,user_id,music_app_user_id,age,education,spending_capacity_band,home_city,stated_favourite_genre,past_purchase_count,first_timer,...,avg_genre_plays,max_genre_plays,days_since_sale_opened,days_until_concert,distance,total_cost,city_match,has_discount,has_music_app,is_travel_search
0,R2F019957,U2CFDCAAD,M1B01D2D7,23,postgraduate,2,Nantes,pop,5,0,...,6.250000,24.0,3.0,72.833333,507.183585,84.50,0,0,1,0
1,R0804D93A,U1FC01010,M1B2C63CC,51,undergraduate,5,Bordeaux,pop,0,1,...,7.333333,10.0,3.0,40.833333,435.686889,95.10,0,0,1,0
2,R367FE81E,U0C605712,M3AEF638D,32,secondary,1,Lyon,classical,1,0,...,1.500000,3.0,3.0,56.833333,435.686889,155.50,0,1,1,0
3,R344BBE29,U25D1DD8A,M00C580ED,24,secondary,2,Paris,jazz_funk,8,0,...,11.500000,27.0,3.0,24.833333,0.000000,94.04,1,0,1,0
4,R004ABDEB,U1600BC92,M29AC37E6,21,secondary,3,Bordeaux,NaN,0,1,...,5.800000,16.0,3.0,88.833333,275.134830,105.70,0,0,1,1
